### Data Cleaning

- Import pandas library
- Load flight data from Excel sheet `flights_dataset_updated_v2.xlsx` (`flights_dataset` sheet)
- Inspect the dataset with `head()`, `info()`, `describe()`
- Check for duplicates with `df[df.duplicated()]`
- Drop unnecessary columns: `checked_bags`, `cabin_bags`
- Convert `departure_time` and `arrival_time` to datetime and split into date and clock time columns:
    - `departure_date`, `departure_clock_time`
    - `arrival_date`, `arrival_clock_time`
- Recalculate `days_until_departure` as `departure_date - query_date`
- Rearrange columns and rename:
    - `day_of_week` → `day_of_week_departure`
    - `month` → `month_departure`
- Load airport location data from the `Airports` sheet
- Merge airport coordinates into the flight data for origin and destination
- Define a Haversine distance function and calculate `distance_km` for each route
- Drop redundant columns to leave the cleaned dataset
- Export the cleaned dataset to `flights_cleaned_df.csv` for EDA

In [1]:
# Import libraries
import pandas as pd

In [2]:
df = pd.read_excel('./flights_dataset_updated_v2.xlsx','flights_dataset')
df.head()

,origin,destination,query_date,departure_time,arrival_time,day_of_week,month,days_until_departure,airline,aircraft,trip_duration_minutes,number_of_stops,currency,base_price,total_price,bookable_seats,checked_bags,cabin_bags
0,YOW,YYZ,2026-03-08,2026-03-09T05:40:00,2026-03-09T07:01:00,0,3,0,WS,7M8,81,0,EUR,216,287.58,7,0,0
1,YOW,YYZ,2026-03-08,2026-03-09T06:45:00,2026-03-09T08:00:00,0,3,0,PD,295,75,0,EUR,216,290.50,9,0,0
2,YOW,YYZ,2026-03-08,2026-03-09T10:00:00,2026-03-09T11:15:00,0,3,0,PD,295,75,0,EUR,216,290.50,9,0,0
3,YOW,YYZ,2026-03-08,2026-03-09T15:05:00,2026-03-09T16:20:00,0,3,0,PD,295,75,0,EUR,216,290.50,9,0,0
4,YOW,YYZ,2026-03-08,2026-03-09T18:55:00,2026-03-09T20:10:00,0,3,0,PD,295,75,0,EUR,216,290.50,9,0,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43242 entries, 0 to 43241
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   origin                 43242 non-null  object        
 1   destination            43242 non-null  object        
 2   query_date             43242 non-null  datetime64[ns]
 3   departure_time         43242 non-null  object        
 4   arrival_time           43242 non-null  object        
 5   day_of_week            43242 non-null  int64         
 6   month                  43242 non-null  int64         
 7   days_until_departure   43242 non-null  int64         
 8   airline                43242 non-null  object        
 9   aircraft               43242 non-null  object        
 10  trip_duration_minutes  43242 non-null  int64         
 11  number_of_stops        43242 non-null  int64         
 12  currency               43242 non-null  object        
 13  b

In [4]:
df.describe()

,query_date,day_of_week,month,days_until_departure,trip_duration_minutes,number_of_stops,base_price,total_price,bookable_seats,checked_bags,cabin_bags
count,43242,43242.000000,43242.000000,43242.0,43242.000000,43242.000000,43242.000000,43242.000000,43242.000000,43242.0,43242.0
mean,2026-03-07 23:59:59.999999744,4.749248,6.177466,0.0,291.460825,0.777670,199.783498,266.358800,7.859812,0.0,0.0
min,2026-03-08 00:00:00,0.000000,1.000000,0.0,64.000000,0.000000,22.000000,46.530000,1.000000,0.0,0.0
25%,2026-03-08 00:00:00,4.000000,5.000000,0.0,229.000000,1.000000,130.000000,187.937500,7.000000,0.0,0.0
50%,2026-03-08 00:00:00,6.000000,6.000000,0.0,325.000000,1.000000,168.000000,231.470000,9.000000,0.0,0.0
75%,2026-03-08 00:00:00,6.000000,8.000000,0.0,367.000000,1.000000,224.000000,300.890000,9.000000,0.0,0.0
max,2026-03-08 00:00:00,6.000000,12.000000,0.0,717.000000,2.000000,2432.000000,2789.940000,9.000000,0.0,0.0
std,NaN,1.961507,2.103253,0.0,118.209078,0.425112,133.289708,148.262115,1.642531,0.0,0.0


In [5]:
# investigate duplicates
df[df.duplicated()].head()

,origin,destination,query_date,departure_time,arrival_time,day_of_week,month,days_until_departure,airline,aircraft,trip_duration_minutes,number_of_stops,currency,base_price,total_price,bookable_seats,checked_bags,cabin_bags
18027,YOW,YVR,2026-03-08,2026-10-25T13:05:00,2026-10-25T21:50:00,6,10,0,WS,73W,395,2,EUR,106,173.25,9,0,0
18031,YOW,YVR,2026-03-08,2026-10-25T05:55:00,2026-10-25T12:35:00,6,10,0,WS,7M8,424,2,EUR,106,180.38,9,0,0
18041,YOW,YVR,2026-03-08,2026-10-25T06:00:00,2026-10-25T16:10:00,6,10,0,WS,7M8,420,2,EUR,106,199.75,9,0,0
18056,YOW,YVR,2026-03-08,2026-10-25T05:55:00,2026-10-25T14:01:00,6,10,0,WS,7M8,415,2,EUR,218,321.92,9,0,0
18057,YOW,YVR,2026-03-08,2026-10-25T05:55:00,2026-10-25T14:01:00,6,10,0,WS,7M8,415,2,EUR,218,321.92,9,0,0


can conclude we have no duplicates as each row returned are unique

In [6]:
# drop checked_bag and cabin_bags
df.drop(['checked_bags','cabin_bags'], axis=1, inplace= True)

- split departure_time - into date and time
- split arrival_time - into date and time

In [7]:
df['departure_time'] = pd.to_datetime(df['departure_time'])
df['arrival_time'] = pd.to_datetime(df['arrival_time'])

df['departure_date'] = df['departure_time'].dt.date
df['departure_clock_time'] = df['departure_time'].dt.time
df['arrival_date'] = df['arrival_time'].dt.date
df['arrival_clock_time'] = df['arrival_time'].dt.time

recalculate days_until_departure

In [8]:
df['query_date'] = pd.to_datetime(df['query_date'])
df['departure_date'] = pd.to_datetime(df['departure_date'])

df['days_until_departure'] = (df['departure_date'] - df['query_date']).dt.days

In [9]:
# rearrage columns
df = df.loc[:,
       ['origin', 'destination', 'airline', 'aircraft', 'query_date'
        ,'departure_date','departure_clock_time', 'day_of_week','month', 'arrival_date','arrival_clock_time'
        , 'days_until_departure','trip_duration_minutes','number_of_stops',
        'bookable_seats','base_price','total_price']]

# rename columns for clarity
df.rename(columns = {'day_of_week' : 'day_of_week_departure',
                     'month' : 'month_departure'}, inplace = True)

In [10]:
# # convert euros to cad
# df['base_price_cad'] = round(df['base_price'] * 1.58, 2)
# df['total_price_cad'] = round(df['total_price'] * 1.58, 2)

# drop base_price and currency
# df.drop(['currency'], axis=1, inplace= True)

In [11]:
df.sample(5)

,origin,destination,airline,aircraft,query_date,departure_date,departure_clock_time,day_of_week_departure,month_departure,arrival_date,arrival_clock_time,days_until_departure,trip_duration_minutes,number_of_stops,bookable_seats,base_price,total_price
18644,YOW,YYC,AC,CR9,2026-03-08,2026-05-24,16:00:00,6,5,2026-05-24,21:30:00,77,332,1,8,161,213.36
29801,YVR,YYZ,AC,CR9,2026-03-08,2026-04-26,08:10:00,6,4,2026-04-26,19:05:00,49,320,1,9,151,175.11
30234,YVR,YYZ,WS,7M8,2026-03-08,2026-05-24,19:30:00,6,5,2026-05-25,06:52:00,77,322,1,7,105,150.00
22560,YYZ,YVR,WS,7M8,2026-03-08,2026-07-12,18:30:00,6,7,2026-07-12,22:35:00,126,355,1,7,137,208.28
29052,YVR,YYZ,WS,DH4,2026-03-08,2026-03-08,05:30:00,6,3,2026-03-08,14:58:00,0,328,1,7,621,675.24


### Data Preparation

we want to calcuate the distance between the two airports given the geographic locations.

In [12]:
# import the airports sheet
df2 = pd.read_excel('./flights_dataset_updated_v2.xlsx','Airports')

# import the airlines sheet
df4 = pd.read_excel('./flights_dataset_updated_v2.xlsx','Airlines')

df4.head()

,ID,Airline,Name
0,1,WS,WestJet
1,2,PD,Porter Airlines Canada Limited
2,3,AC,Air Canada
3,4,TS,Air Transat
4,5,5T,First Air Canadian North


join the locations to the origin and desinations

In [13]:
df3 = df.copy()

# join tables
df3 = df3.merge(df2, left_on='origin', right_on='Code', how='left', suffixes=('', '_origin'))
df3 = df3.merge(df2, left_on='destination', right_on='Code', how='left', suffixes=('', '_destination'))

# add the airline names to the main dataframe
df3 = df3.merge(df4, left_on='airline', right_on='Airline', how='left', suffixes=('', '_airline'))

#previw data
df3.sample(3)

,origin,destination,airline,aircraft,query_date,departure_date,departure_clock_time,day_of_week_departure,month_departure,arrival_date,...,x,y,Code_destination,City_destination,Name_destination,x_destination,y_destination,ID,Airline,Name_airline
13183,YOW,YVR,AC,320,2026-03-08,2026-08-11,20:15:00,1,8,2026-08-12,...,45.3233,-75.6667,YVR,Vancouver,Vancouver International,49.1950,-123.1833,3,AC,Air Canada
33925,YVR,YYC,WS,7M8,2026-03-08,2026-06-14,20:30:00,6,6,2026-06-14,...,49.1950,-123.1833,YYC,Calgary,Calgary International,51.1133,-114.0200,1,WS,WestJet
18340,YOW,YYC,AC,DH4,2026-03-08,2026-04-05,19:35:00,6,4,2026-04-06,...,45.3233,-75.6667,YYC,Calgary,Calgary International,51.1133,-114.0200,3,AC,Air Canada


function to calculate the distance between two airports routes

In [14]:
from math import radians, cos, sin, asin, sqrt

def distance(lat1, lon1, lat2, lon2):

    # Convert latitude and longitude from degrees to radians
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371  # Radius of Earth in kilometers
    return c * r

# apply the distnace function to the dataframe
df3['distance_km'] = df3.apply(lambda row: distance(row['x'], row['y'], row['x_destination'], row['y_destination']), axis=1)

In [15]:
# drop redundant columns
df3 = df3.loc[:,['origin','City','destination','City_destination','distance_km','Name_airline', 'aircraft', 'query_date',
       'departure_date', 'departure_clock_time', 'day_of_week_departure',
       'month_departure', 'arrival_date', 'arrival_clock_time',
       'days_until_departure', 'trip_duration_minutes', 'number_of_stops',
       'bookable_seats', 'base_price']]

df3.sample(5)

,origin,City,destination,City_destination,distance_km,Name_airline,aircraft,query_date,departure_date,departure_clock_time,day_of_week_departure,month_departure,arrival_date,arrival_clock_time,days_until_departure,trip_duration_minutes,number_of_stops,bookable_seats,base_price
4741,YOW,Ottawa,YYZ,Toronto,363.703036,Porter Airlines Canada Limited,295,2026-03-08,2026-12-30,17:25:00,2,12,2026-12-30,18:40:00,297,75,0,9,137
24381,YYZ,Toronto,YYC,Calgary,2689.246727,Air Canada,789,2026-03-08,2026-03-15,08:00:00,6,3,2026-03-15,13:55:00,7,385,1,9,412
5239,YOW,Ottawa,YVR,Vancouver,3552.042539,Air Canada,CR9,2026-03-08,2026-03-18,10:50:00,2,3,2026-03-18,16:11:00,10,396,1,9,195
19449,YOW,Ottawa,YYC,Calgary,2878.682577,WestJet,73W,2026-03-08,2026-09-20,13:05:00,6,9,2026-09-20,18:40:00,196,300,1,7,128
30890,YVR,Vancouver,YYZ,Toronto,3345.516314,Air Canada,DH4,2026-03-08,2026-06-28,21:55:00,6,6,2026-06-29,06:59:00,112,308,1,5,270


### Export data for EDA

In [16]:
df3.to_csv('./flights_cleaned_df.csv', index=False)